# IPL Data Cleaning, Analysis, and Multi-Model Training

This notebook does four things:
1. Loads your IPL dataset
2. Cleans and prepares data
3. Performs quick EDA
4. Trains and compares several models

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

In [ ]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
candidates = [
    ROOT / 'ipl_colab.csv',
    ROOT / 'data' / 'processed' / 'ipl_features.csv'
]

data_path = None
for p in candidates:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError('No dataset found. Expected ipl_colab.csv or data/processed/ipl_features.csv')

df_raw = pd.read_csv(data_path, low_memory=False)
print('Loaded:', data_path)
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.drop_duplicates()

    for col in out.columns:
        if 'date' in col.lower():
            parsed = pd.to_datetime(out[col], errors='coerce')
            if parsed.notna().sum() > 0:
                out[col] = parsed

    for col in out.select_dtypes(include=['object']).columns:
        out[col] = out[col].astype(str).str.strip()
        out[col] = out[col].replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

    return out

df = clean_dataframe(df_raw)
print('Raw shape:', df_raw.shape)
print('Clean shape:', df.shape)
df.head()

## Quick EDA

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing.to_frame('missing_count').head(20)

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print('Numeric columns:', len(numeric_cols))
df[numeric_cols].describe().T.head(20) if numeric_cols else 'No numeric columns found'

In [ ]:
if numeric_cols:
    plot_col = numeric_cols[0]
    plt.figure(figsize=(8, 4))
    df[plot_col].dropna().hist(bins=30)
    plt.title(f'Distribution of {plot_col}')
    plt.xlabel(plot_col)
    plt.ylabel('Frequency')
    plt.show()
else:
    print('Skipping histogram: no numeric columns.')

## Model Utilities

In [ ]:
def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=['number', 'bool', 'datetime64[ns]']).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', num_pipe, numeric_cols),
        ('cat', cat_pipe, categorical_cols)
    ])
    return preprocessor

def run_regression(df_in: pd.DataFrame, target: str) -> pd.DataFrame:
    frame = df_in.dropna(subset=[target]).copy()
    X = frame.drop(columns=[target])
    y = frame[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    prep = build_preprocessor(X_train)

    models = {
        'LinearRegression': LinearRegression(),
        'RandomForestRegressor': RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1),
        'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42)
    }

    rows = []
    for name, model in models.items():
        pipe = Pipeline([('prep', prep), ('model', model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)

        rows.append({
            'Model': name,
            'MAE': float(mean_absolute_error(y_test, pred)),
            'RMSE': float(mean_squared_error(y_test, pred) ** 0.5),
            'R2': float(r2_score(y_test, pred))
        })

    return pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)

def run_classification(df_in: pd.DataFrame, target: str) -> pd.DataFrame:
    frame = df_in.dropna(subset=[target]).copy()
    X = frame.drop(columns=[target])
    y = frame[target]

    if y.nunique() < 2:
        raise ValueError('Classification target has fewer than 2 classes after cleaning.')

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
    )
    prep = build_preprocessor(X_train)

    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'RandomForestClassifier': RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1),
        'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42)
    }

    rows = []
    binary = y.nunique() == 2

    for name, model in models.items():
        pipe = Pipeline([('prep', prep), ('model', model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)

        row = {
            'Model': name,
            'Accuracy': float(accuracy_score(y_test, pred)),
            'F1_weighted': float(f1_score(y_test, pred, average='weighted'))
        }

        if binary and hasattr(pipe.named_steps['model'], 'predict_proba'):
            prob = pipe.predict_proba(X_test)[:, 1]
            row['ROC_AUC'] = float(roc_auc_score(y_test, prob))

        rows.append(row)

    return pd.DataFrame(rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)

## Regression: Predict Total Runs

In [ ]:
reg_target = 'total' if 'total' in df.columns else ('total_runs' if 'total_runs' in df.columns else None)

if reg_target is None:
    print('No standard regression target found (expected total or total_runs).')
else:
    reg_results = run_regression(df, reg_target)
    display(reg_results)
    print('Best regression model:', reg_results.iloc[0]['Model'])

## Classification: Predict Win (or Derived Label)

In [ ]:
if 'win' in df.columns:
    clf_df = df.copy()
    clf_target = 'win'
else:
    clf_df = df.copy()
    if reg_target is None:
        raise ValueError('No win column and no regression target to derive a class label from.')
    threshold = clf_df[reg_target].median()
    clf_df['high_score_label'] = (clf_df[reg_target] >= threshold).astype(int)
    clf_target = 'high_score_label'

clf_results = run_classification(clf_df, clf_target)
display(clf_results)
print('Best classification model:', clf_results.iloc[0]['Model'])